# ==========================================================
# Logistic Regression Model Development
# ==========================================================

"""
Objective

Develop a Logistic Regression model for predicting
Composite Cardiovascular Disease using the final
feature-selected dataset.

Workflow

1. Load Dataset
2. Verify Dataset
3. Handle Class Imbalance
4. Train Model
5. Evaluate Model
6. Interpret Model
7. Save Outputs
"""

In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Models
from sklearn.linear_model import LogisticRegression

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    cohen_kappa_score,
    RocCurveDisplay
)

from sklearn.metrics import roc_curve, auc

# Handle Imbalanced Data
from imblearn.over_sampling import SMOTE

from explainerdashboard import ClassifierExplainer
from explainerdashboard import ExplainerDashboard

# Save Model
import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
# Load Final Dataset

output_path = "C:/Users/kalya/NTCC/NHANES/CVD_ML_python/Outputs/"

X_train = pd.read_csv(output_path + "X_train_final.csv")
X_test = pd.read_csv(output_path + "X_test_final.csv")

y_train = pd.read_csv(
    output_path + "y_train_final.csv"
).squeeze("columns")

y_test = pd.read_csv(
    output_path + "y_test_final.csv"
).squeeze("columns")

print("Datasets Loaded Successfully")

print("\nTraining Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

Datasets Loaded Successfully

Training Shape : (12365, 44)
Testing Shape  : (3092, 44)


In [3]:

# Verify Dataset

print("="*60)
print("FINAL DATASET SUMMARY")
print("="*60)

print("Training Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])

print("Features :", X_train.shape[1])

print("\nMissing Values")

print("Training :", X_train.isna().sum().sum())
print("Testing  :", X_test.isna().sum().sum())

print("\nTarget Distribution")

print("\nTraining")

print(round(y_train.value_counts(normalize=True)*100,2))

print("\nTesting")

print(round(y_test.value_counts(normalize=True)*100,2))

FINAL DATASET SUMMARY
Training Samples : 12365
Testing Samples  : 3092
Features : 44

Missing Values
Training : 0
Testing  : 0

Target Distribution

Training
Composite_CVD
0    92.71
1     7.29
Name: proportion, dtype: float64

Testing
Composite_CVD
0    92.72
1     7.28
Name: proportion, dtype: float64


In [4]:
# ==========================================================
# Handle Class Imbalance Using SMOTE
# ==========================================================

smote = SMOTE(
    random_state=42
)

X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train,
    y_train
)

print("="*60)
print("AFTER SMOTE")
print("="*60)

print("Training Shape :", X_train_balanced.shape)

print("\nTarget Distribution")

print(y_train_balanced.value_counts())

AFTER SMOTE
Training Shape : (22928, 44)

Target Distribution
Composite_CVD
0    11464
1    11464
Name: count, dtype: int64


In [5]:
# ==========================================================
# Logistic Regression
# ==========================================================

# Create Model
logistic_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

# Train Model
logistic_model.fit(
    X_train_balanced,
    y_train_balanced
)

print("Logistic Regression Training Completed")

Logistic Regression Training Completed


In [6]:
# ==========================================================
# Logistic Regression Predictions
# ==========================================================

y_pred_lr = logistic_model.predict(X_test)

y_prob_lr = logistic_model.predict_proba(X_test)[:,1]

In [7]:
# ==========================================================
# Logistic Regression Performance
# ==========================================================

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)
lr_mcc = matthews_corrcoef(y_test, y_pred_lr)
lr_kappa = cohen_kappa_score(y_test, y_pred_lr)

print("="*60)
print("LOGISTIC REGRESSION PERFORMANCE")
print("="*60)

print(f"Accuracy        : {lr_accuracy:.4f}")
print(f"Precision       : {lr_precision:.4f}")
print(f"Recall          : {lr_recall:.4f}")
print(f"F1 Score        : {lr_f1:.4f}")
print(f"ROC AUC         : {lr_auc:.4f}")
print(f"Matthews CC     : {lr_mcc:.4f}")
print(f"Cohen's Kappa   : {lr_kappa:.4f}")

LOGISTIC REGRESSION PERFORMANCE
Accuracy        : 0.8503
Precision       : 0.2920
Recall          : 0.7422
F1 Score        : 0.4191
ROC AUC         : 0.8992
Matthews CC     : 0.4020
Cohen's Kappa   : 0.3513


In [8]:
# ==========================================================
# Classification Report
# ==========================================================

print(classification_report(
    y_test,
    y_pred_lr
))

              precision    recall  f1-score   support

           0       0.98      0.86      0.91      2867
           1       0.29      0.74      0.42       225

    accuracy                           0.85      3092
   macro avg       0.63      0.80      0.67      3092
weighted avg       0.93      0.85      0.88      3092



In [9]:
# ==========================================================
# Explainable AI Dashboard
# ==========================================================

explainer = ClassifierExplainer(
    logistic_model,
    X_test,
    y_test
)

dashboard = ExplainerDashboard(explainer)

print("Explainer Created Successfully")

Background dataset has 3092 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=3092 when initializing the masker.


Explainer Created Successfully


In [10]:
# ==========================================================
# Save Dashboard as HTML
# ==========================================================

html = dashboard.to_html()

with open(
    output_path + "Random_Forest_Dashboard.html",
    "w",
    encoding="utf-8"
) as f:
    f.write(html)

print("Dashboard Saved Successfully")

Dashboard Saved Successfully


In [11]:
import os

model_path = "C:/Users/kalya/NTCC/NHANES/CVD_ML_python/Models/"

os.makedirs(model_path, exist_ok=True)

# ==========================================================
# Save Logistic Regression Model
# ==========================================================

joblib.dump(
    logistic_model,
    model_path + "Logistic_Regression_Model.pkl"
)

print("Logistic Regression Model Saved Successfully")

Logistic Regression Model Saved Successfully


In [12]:
print("="*70)
print("LOGISTIC REGRESSION MODEL COMPLETED")
print("="*70)

print(f"Training Samples : {X_train_balanced.shape[0]}")
print(f"Testing Samples  : {X_test.shape[0]}")
print(f"Selected Features: {X_train.shape[1]}")
print(f"ROC-AUC          : {lr_auc:.4f}")
print(f"F1 Score         : {lr_f1:.4f}")

print("\nAll outputs have been saved successfully.")

LOGISTIC REGRESSION MODEL COMPLETED
Training Samples : 22928
Testing Samples  : 3092
Selected Features: 44
ROC-AUC          : 0.8992
F1 Score         : 0.4191

All outputs have been saved successfully.
